# Rio Controller API - Jupyter Notebook Example

This notebook demonstrates how to control the Rio controller from a Jupyter notebook using the REST API and WebSocket streaming interface.

**Note:** The API now uses **LabThings/WoT-compliant Things**. This notebook uses the legacy `/api/control/*` routes for backward compatibility. WoT routes are available at `/flow/`, `/heater/`, `/camera/`, etc. (see API docs at `/docs`).

## Prerequisites

1. Rio API server must be running (see `software/api/README.md`)
2. Install required packages:
   ```bash
   pip install requests websocket-client matplotlib pandas numpy
   ```

## Configuration

Update the `API_BASE_URL` below to match your Rio controller's IP address and port.


In [1]:
# Configuration
import os

# Set API base URL (can also use environment variable RIO_API_URL)
# For local development: "http://localhost:8000"
# For Raspberry Pi: "http://raspberrypi.local:8000" or "http://192.168.1.100:8000"
API_BASE_URL = os.getenv("RIO_API_URL", "http://localhost:8000")
# Uncomment and update for Pi access:
# API_BASE_URL = "http://raspberrypi.local:8000"

import sys
from pathlib import Path

# Add software/client to path for client import
# This works whether running from repo root or software/client/notebooks/
repo_root = Path.cwd()
if (repo_root / "software" / "client").exists():
    # Running from repo root
    sys.path.insert(0, str(repo_root / "software"))
elif (repo_root.parent.parent / "software" / "client").exists():
    # Running from software/client/notebooks/
    sys.path.insert(0, str(repo_root.parent.parent))
else:
    # Fallback: try to find software directory
    for parent in repo_root.parents:
        if (parent / "software" / "client").exists():
            sys.path.insert(0, str(parent / "software"))
            break

from client import RioClient, RioStreamClient
import requests
import json
import time
from datetime import datetime
from IPython.display import display, Image, HTML, clear_output
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Initialize client
client = RioClient(base_url=API_BASE_URL)

print("✅ Rio API client initialized")
print(f"   Base URL: {API_BASE_URL}")


✅ Rio API client initialized
   Base URL: http://localhost:8000


## 1. System Health and Capabilities

Check if the API server is running and what modules are available.


In [2]:
# Check health
health = client.health()
print(f"Status: {health['status']}")
print(f"Simulation mode: {health['simulation']}")

# Get capabilities
capabilities = client.capabilities()
print("\nAvailable modules:")
for module, available in capabilities['modules'].items():
    status = "✅" if available else "❌"
    print(f"  {status} {module}: {available}")


Status: ok
Simulation mode: True

Available modules:
  ✅ flow: True
  ✅ pressure: True
  ✅ heater: True
  ❌ strobe: False
  ❌ camera: False
  ❌ droplet: False
  ❌ pump: False


## 2. Channel Configuration

View and update channel metadata (names, liquid types, calibration factors).


In [3]:
# Get current channel configuration
channels = client.get_channels()
print("Current channel configuration:")
print(json.dumps(channels, indent=2))

# Example: Update channel name and liquid type
# Uncomment to update:
# update_config = {
#     "flow": {
#         "0": {
#             "name": "oil",
#             "liquid_type": "mineral_oil",
#             "enabled": True
#         }
#     }
# }
# result = client.set_channels(update_config)
# print("Updated channel configuration")


Current channel configuration:
{
  "channels": {
    "flow": {
      "0": {
        "enabled": true,
        "name": "",
        "liquid_type": "",
        "calibration_factor": 1.0
      },
      "1": {
        "enabled": true,
        "name": "",
        "liquid_type": "",
        "calibration_factor": 1.0
      },
      "2": {
        "enabled": true,
        "name": "",
        "liquid_type": "",
        "calibration_factor": 1.0
      },
      "3": {
        "enabled": true,
        "name": "",
        "liquid_type": "",
        "calibration_factor": 1.0
      }
    },
    "pressure": {
      "0": {
        "enabled": true,
        "name": "",
        "liquid_type": "",
        "calibration_factor": 1.0
      },
      "1": {
        "enabled": true,
        "name": "",
        "liquid_type": "",
        "calibration_factor": 1.0
      },
      "2": {
        "enabled": true,
        "name": "",
        "liquid_type": "",
        "calibration_factor": 1.0
      },
      "3": {
    

## 3. Flow/Pressure Control

Control flow and pressure channels.


In [4]:
# Get current flow/pressure state
state = client.get_flow_state()
print("Current flow/pressure state:")
print(f"  Pressure targets (mbar): {state['pressure_targets_mbar']}")
print(f"  Pressure actuals (mbar): {state['pressure_actuals_mbar']}")
print(f"  Flow targets (ul/hr): {state['flow_targets_ul_hr']}")
print(f"  Flow actuals (ul/hr): {state['flow_actuals_ul_hr']}")
print(f"  Control modes: {state['control_modes_text']}")

# Example: Set flow rate for channel 0
# Uncomment to set:
# client.set_flow(0, 100.0)  # Set channel 0 to 100 ul/hr
# print("Set channel 0 flow to 100 ul/hr")

# Example: Set pressure for channel 1
# client.set_pressure(1, 50.0)  # Set channel 1 to 50 mbar
# print("Set channel 1 pressure to 50 mbar")


Current flow/pressure state:
  Pressure targets (mbar): [0.0, 0.0, 0.0, 0.0]
  Pressure actuals (mbar): [0.0, 0.625, 4.375, 1.25]
  Flow targets (ul/hr): [0.0, 0.0, 0.0, 0.0]
  Flow actuals (ul/hr): [1.0, 0.0, 0.0, 2.0]
  Control modes: ['Off', 'Off', 'Off', 'Off']


## 4. Heater Control

Control heater temperature and PID settings.


In [5]:
# Get heater states
heater_state = client.get_heater_state()
print("Heater states:")
for i, heater in enumerate(heater_state['heaters']):
    print(f"  Heater {i}:")
    print(f"    Actual temp: {heater['temp_c_actual']:.1f}°C")
    print(f"    Target temp: {heater['temp_c_target']:.1f}°C")
    print(f"    PID enabled: {heater['pid_enabled']}")
    print(f"    Stir enabled: {heater['stir_enabled']}")
    print(f"    Status: {heater['status_text']}")

# Example: Set heater temperature
# Uncomment to set:
# client.set_heater_temp(0, 37.0)  # Set heater 0 to 37°C
# print("Set heater 0 target to 37°C")

# Example: Enable PID control
# client.set_heater_pid(0, True)
# print("Enabled PID for heater 0")


Heater states:
  Heater 0:
    Actual temp: 25.0°C
    Target temp: 25.0°C
    PID enabled: False
    Stir enabled: False
    Status: Unconfigured
  Heater 1:
    Actual temp: 25.0°C
    Target temp: 25.0°C
    PID enabled: False
    Stir enabled: False
    Status: Unconfigured
  Heater 2:
    Actual temp: 25.0°C
    Target temp: 25.0°C
    PID enabled: False
    Stir enabled: False
    Status: Unconfigured
  Heater 3:
    Actual temp: 25.0°C
    Target temp: 25.0°C
    PID enabled: False
    Stir enabled: False
    Status: Unconfigured


## 5. Camera Snapshot

Capture a single frame from the camera.


In [6]:
# Get camera snapshot
try:
    snapshot = client.get_camera_snapshot()
    display(Image(data=snapshot, width=400))
    print("✅ Camera snapshot captured")
except Exception as e:
    print(f"❌ Failed to get snapshot: {e}")


❌ Failed to get snapshot: Failed to get camera snapshot: 503 Server Error: Service Unavailable for url: http://localhost:8000/api/streams/camera/snapshot


## 6. WebSocket Telemetry Streaming

Stream real-time sensor data (flow, pressure, heater) via WebSocket.


In [7]:
# Initialize stream client
stream_client = RioStreamClient(base_url=API_BASE_URL)

# Subscribe to topics and channels
stream_client.subscribe(
    topics=["flow", "pressure", "heater"],
    channels={"flow": [0, 1], "pressure": [0, 1], "heater": [0]}  # Only specific channels
)

# Collect messages for a few seconds
print("Collecting telemetry data (5 seconds)...")
messages = []
start_time = time.time()

for msg in stream_client.iter_messages(timeout=5.0):
    messages.append(msg)
    if len(messages) % 10 == 0:
        print(f"  Received {len(messages)} messages...")

print(f"\n✅ Collected {len(messages)} messages")

# Display sample messages
if messages:
    print("\nSample messages:")
    for msg in messages[:5]:
        print(f"  {msg}")


WebSocket error: Handshake status 403 Forbidden -+-+- {'date': 'Tue, 13 Jan 2026 17:23:26 GMT', 'content-length': '0', 'content-type': 'text/plain', 'connection': 'close'} -+-+- b''


RioWebSocketError: Failed to connect to ws://localhost:8000/api/streams/aggregate within 5.0s

## 7. Plot Streaming Data

Visualize the streamed sensor data.


In [ ]:
if messages:
    # Convert messages to DataFrame for easier plotting
    df = pd.DataFrame(messages)
    
    # Plot flow data
    flow_data = df[df['topic'] == 'flow']
    if not flow_data.empty:
        fig, axes = plt.subplots(2, 1, figsize=(10, 6))
        
        # Plot flow values by channel
        for channel in flow_data['channel'].unique():
            channel_data = flow_data[flow_data['channel'] == channel]
            axes[0].plot(channel_data['timestamp'], channel_data['value'], 
                        label=f"Channel {channel}", marker='o', markersize=2)
        axes[0].set_xlabel('Time (s)')
        axes[0].set_ylabel('Flow (ul/hr)')
        axes[0].set_title('Flow Rate Over Time')
        axes[0].legend()
        axes[0].grid(True)
        
        # Plot pressure values by channel
        pressure_data = df[df['topic'] == 'pressure']
        for channel in pressure_data['channel'].unique():
            channel_data = pressure_data[pressure_data['channel'] == channel]
            axes[1].plot(channel_data['timestamp'], channel_data['value'],
                        label=f"Channel {channel}", marker='o', markersize=2)
        axes[1].set_xlabel('Time (s)')
        axes[1].set_ylabel('Pressure (mbar)')
        axes[1].set_title('Pressure Over Time')
        axes[1].legend()
        axes[1].grid(True)
        
        plt.tight_layout()
        plt.show()
    else:
        print("No flow data to plot")
else:
    print("No messages collected")


## 8. On-Demand Data Capture

Start CSV capture of sensor data for later analysis.


In [ ]:
# Start data capture
capture_path = f"capture_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
capture_status = client.capture_start(
    topics=["flow", "pressure"],
    channels={"flow": [0, 1], "pressure": [0, 1]},
    path=capture_path
)

print(f"✅ Capture started: {capture_status['enabled']}")
print(f"   Path: {capture_status['path']}")
print(f"   Topics: {capture_status['topics']}")

# Let it run for a few seconds
print("\nCapturing data for 5 seconds...")
time.sleep(5)

# Check status
status = client.capture_status()
print(f"\nCapture status: {status}")

# Stop capture
stop_status = client.capture_stop()
print(f"\n✅ Capture stopped: {stop_status['enabled']}")
print(f"   Data saved to: {stop_status['path']}")


## 9. Droplet Detection (if available)

Control droplet detection and view statistics.


In [ ]:
# Re-fetch capabilities (defined in cell 3)
capabilities = client.capabilities()

# Check if droplet detection is available
if capabilities['modules'].get('droplet', False):
    # Get status
    status = client.droplet_status()
    print("Droplet detection status:")
    print(f"  Running: {status['running']}")
    print(f"  Frame count: {status['frame_count']}")
    print(f"  Droplet count: {status['droplet_count_total']}")
    print(f"  Processing rate: {status['processing_rate_hz']:.2f} Hz")
    
    # Get statistics
    stats = client.droplet_statistics()
    print("\nStatistics:")
    print(json.dumps(stats, indent=2))
    
    # Example: Start detection
    # client.droplet_start()
    # print("Started droplet detection")
else:
    print("❌ Droplet detection not available")


## Notes

- All API endpoints are documented in the OpenAPI/Swagger UI at `http://<API_BASE_URL>/docs`
- Channel metadata updates (names, liquid types) are runtime-only and not persisted
- Calibration factors are loaded from the main config YAML file (`rio-config.yaml`)
- WebSocket streaming supports high-rate data (20-50 Hz) with optional client-side decimation
- Data capture saves CSV files with timestamps, channel indices, and calibrated values
